In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")
from time import time
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
data=pd.read_excel(r"D:\projects\activation_energy_prediction\G_models\Ga DelG dataset for prediction.xlsx")

In [35]:
num_reactions = 153
# Create an array of indices representing the reactions
reaction_indices = np.arange(num_reactions)

# Split the reactions into train and test sets
train_indices, test_indices = train_test_split(reaction_indices, test_size=0.3,random_state=42)

# Initialize empty lists to hold train and test data
train_data=pd.DataFrame()
test_data =pd.DataFrame()

# Populate train and test data using the selected indices

for i in range(51):
    for idx in train_indices:
        train_data=train_data._append(data.iloc[idx+num_reactions*i],ignore_index=True)
    for idx in test_indices:
        test_data=test_data._append(data.iloc[idx+num_reactions*i],ignore_index=True)

In [36]:
## Ga Model
train=train_data.drop(['Unnamed: 0',"Reactions","Reactant 1","Reactant 2","Product 1","Product 2","Del G","Del E"],axis=1)
test=test_data.drop(['Unnamed: 0',"Reactions","Reactant 1","Reactant 2","Product 1","Product 2","Del G","Del E"],axis=1)
train_Y=train[['Ga']]
train_X=train.drop(["Ga"],axis=1)
test_Y=test[['Ga']]
test_X=test.drop(["Ga"],axis=1)
cor = train_X.corr('pearson')
# get upper triangle of correlation matrix
upper = cor.where(np.triu(np.ones(cor.shape), k=1).astype(np.bool_))

# find features with correlation greater than 0.90
to_drop = [column for column in upper.columns if any(upper[column] > 0.90)]

# drop highly correlated features
train_X.drop(to_drop, axis=1, inplace=True)
test_X.drop(to_drop, axis=1, inplace=True)

In [37]:
from sklearn.model_selection import KFold
model=Ridge(alpha=9.958498584649295)

# Perform K-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results_df = pd.DataFrame(columns=['Model', 'Fold', 'RMSE','R2','MAE'])

for fold, (train_idx, val_idx) in enumerate(kf.split(train_X), 1):
    X_train, X_val = train_X.iloc[train_idx], train_X.iloc[val_idx]
    y_train, y_val = train_Y.iloc[train_idx], train_Y.iloc[val_idx]

    # Train Ridge Regressor
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    r2 = r2_score(y_val,pred)
    mae= mean_absolute_error(y_val,pred)
    results_df = results_df._append({'Model': 'Ridge Regressor', 'Fold': fold, 'RMSE': rmse , 'R2': r2 ,'MAE': mae}, ignore_index=True)

# Calculate average RMSE across folds
average_rmse = results_df[results_df['Model'] == 'Ridge Regressor']['RMSE'].mean()
average_r2 = results_df[results_df['Model'] == 'Ridge Regressor']['R2'].mean()
average_mae = results_df[results_df['Model'] == 'Ridge Regressor']['MAE'].mean()

print(f"Average RMSE : {average_rmse:.4f}")
print(f"Average MAE : {average_mae:.4f}")
print(f"Average R2 : {average_r2:.4f}")


# Display results DataFrame
print("\nResults DataFrame:")
print(results_df)

Average RMSE : 0.0536
Average MAE : 0.0382
Average R2 : 0.9794

Results DataFrame:
             Model Fold      RMSE        R2       MAE
0  Ridge Regressor    1  0.052347  0.981387  0.037716
1  Ridge Regressor    2  0.053654  0.980258  0.038466
2  Ridge Regressor    3  0.055995  0.979289  0.039250
3  Ridge Regressor    4  0.053186  0.978445  0.037460
4  Ridge Regressor    5  0.052728  0.977626  0.038342


In [38]:
test_pred=model.predict(test_X)
rmse = np.sqrt(mean_squared_error(test_Y, test_pred))
r2 = r2_score(test_Y,test_pred)
mae= mean_absolute_error(test_Y,test_pred)
print(f"RMSE : {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2 : {r2:.4f}")


RMSE : 0.0627
MAE : 0.0491
R2 : 0.9775
